# Graph Inspection Notebook

This notebook loads and inspects the knowledge graph using the utility functions from `cli/commands/evals/utils.py`.

In [1]:
from pathlib import Path
import sys
import polars as pl
import random

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

from cli.commands.evals.utils import (
    load_graph,
    load_node_metadata,
    compute_pagerank,
    pagerank_to_dataframe,
)

17:28:46 - LiteLLM:WARNING: get_model_cost_map.py:271 - LiteLLM: Failed to fetch remote model cost map from : Request URL is missing an 'http://' or 'https://' protocol.. Falling back to local backup.


[03/24/26 17:28:46] INFO     Using 'conf/logging.yml' as logging configuration. You can change this ]8;id=135531;file:///Users/ayush/Documents/Zitnik_Lab/optimus/.venv/lib/python3.13/site-packages/kedro/framework/project/__init__.py\__init__.py]8;;\:]8;id=369643;file:///Users/ayush/Documents/Zitnik_Lab/optimus/.venv/lib/python3.13/site-packages/kedro/framework/project/__init__.py#270\270]8;;\
                             by setting the KEDRO_LOGGING_CONFIG environment variable accordingly.                 

## Load Graph

In [2]:
nodes_path = Path("data/gold/kg/parquet/nodes.parquet")
edges_path = Path("data/gold/kg/parquet/edges.parquet")

G, edge_type_lookup, node_types, edge_types = load_graph(nodes_path, edges_path)

[03/24/26 17:28:47] INFO     Loaded 192035 nodes with 10 node types: ['ANA', 'BPO', 'CCO', 'DIS',       ]8;id=418840;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py\utils.py]8;;\:]8;id=431116;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py#42\42]8;;\
                             'DRG', 'EXP', 'GEN', 'MFN', 'PHE', 'PWY']                                             

[03/24/26 17:29:34] INFO     Loaded 40983766 edges with 26 edge types: ['ANA-ANA', 'ANA-PRO',           ]8;id=857394;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py\utils.py]8;;\:]8;id=733382;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py#66\66]8;;\
                             'BPO-BPO', 'BPO-PRO', 'CCO-CCO', 'CCO-PRO', 'DIS-DIS', 'DIS-PHE',                     
                             'DIS-PRO', 'DRG-DIS', 'DRG-DRG', 'DRG-PHE', 'DRG-PRO', 'EXP-BPO',                     
                             'EXP-CCO', 'EXP-DIS', 'EXP-EXP', 'EXP-MFN', 'EXP-PRO', 'MFN-MFN',                     
                             'MFN-PRO', 'PHE-PHE', 'PHE-PRO', 'PRO-PRO', 'PWY-PRO', 'PWY-PWY']                     

## Basic Statistics

In [3]:
print(f"Nodes: {G.number_of_nodes():,}")
print(f"Edges: {G.number_of_edges():,}")
print(f"\nNode types ({len(node_types)}): {node_types}")
print(f"\nEdge types ({len(edge_types)}): {edge_types}")

Nodes: 192,035
Edges: 40,983,766

Node types (10): ['ANA', 'BPO', 'CCO', 'DIS', 'DRG', 'EXP', 'GEN', 'MFN', 'PHE', 'PWY']

Edge types (26): ['ANA-ANA', 'ANA-PRO', 'BPO-BPO', 'BPO-PRO', 'CCO-CCO', 'CCO-PRO', 'DIS-DIS', 'DIS-PHE', 'DIS-PRO', 'DRG-DIS', 'DRG-DRG', 'DRG-PHE', 'DRG-PRO', 'EXP-BPO', 'EXP-CCO', 'EXP-DIS', 'EXP-EXP', 'EXP-MFN', 'EXP-PRO', 'MFN-MFN', 'MFN-PRO', 'PHE-PHE', 'PHE-PRO', 'PRO-PRO', 'PWY-PRO', 'PWY-PWY']


## Load Node Metadata

In [4]:
node_metadata = load_node_metadata(nodes_path)
node_metadata.head(10)

[03/24/26 17:29:39] INFO     Loaded metadata for 192035 nodes                                          ]8;id=826027;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py\utils.py]8;;\:]8;id=416923;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py#103\103]8;;\

id,label,name
str,str,str
"""ENSG00000000003""","""GEN""","""tetraspanin 6"""
"""ENSG00000000005""","""GEN""","""tenomodulin"""
"""ENSG00000000419""","""GEN""","""dolichyl-phosphate mannosyltra…"
"""ENSG00000000457""","""GEN""","""SCY1 like pseudokinase 3"""
"""ENSG00000000460""","""GEN""","""FIGNL1 interacting regulator o…"
"""ENSG00000000938""","""GEN""","""FGR proto-oncogene, Src family…"
"""ENSG00000000971""","""GEN""","""complement factor H"""
"""ENSG00000001036""","""GEN""","""alpha-L-fucosidase 2"""
"""ENSG00000001084""","""GEN""","""glutamate-cysteine ligase cata…"


In [5]:
# Node counts by type
node_metadata.group_by("label").len().sort("len", descending=True)

label,len
str,u32
"""GEN""",61306
"""DIS""",36345
"""BPO""",25754
"""PHE""",19341
"""DRG""",16766
"""ANA""",14624
"""MFN""",10161
"""CCO""",4052
"""PWY""",2805


## Check Ghost Nodes

In [6]:
# Find node where id == GO_0009199 or GO_0009161
selected_ids = ["GO_0009199", "GO_0009161"]
selected_nodes = node_metadata.filter(pl.col("id").is_in(selected_ids))

In [7]:
selected_ids = ["GO_0009199", "GO_0009161"]
nodes_path = Path("data/gold/kg/parquet/nodes.parquet")
df = pl.read_parquet(nodes_path)
df.filter(pl.col("id").is_in(selected_ids))

id,label,properties
str,str,str


In [8]:
edges_path = Path("data/gold/kg/parquet/edges.parquet")
edges_df = pl.read_parquet(edges_path)
edges_df.filter(pl.col("from").is_in(selected_ids) | pl.col("to").is_in(selected_ids))

from,to,label,relation,undirected,properties
str,str,str,str,bool,str


In [9]:
# Read in pagerank df
pagerank_df = pl.read_csv("data/gold/evals/pagerank.csv")

# Filter for selected nodes
pagerank_df.filter(pl.col("id").is_in(selected_ids))

rank,id,label,name,pagerank
i64,str,str,str,f64
29027,"""GO_0009199""",null,null,0.000005
40158,"""GO_0009161""",null,null,0.000003


## Debug Difference

In [10]:
nodes_path = Path("data/gold/kg/parquet/nodes.parquet")
df = pl.read_parquet(nodes_path)

print("Rows:", df.height)
print("Unique IDs:", df.select(pl.col("id").n_unique()).item())
print("Duplicate rows by ID:", df.height - df.select(pl.col("id").n_unique()).item())

Rows: 192035
Unique IDs: 192035
Duplicate rows by ID: 0


In [11]:
# Identify IDs that appear more than once
duplicate_ids = (
    df.group_by("id")
    .len()
    .filter(pl.col("len") > 1)
    .select("id")
)

# Join back to original data to get full metadata for those IDs
duplicated_nodes_table = (
    df.join(duplicate_ids, on="id", how="inner")
    .with_columns(
        pl.col("properties").str.json_path_match("$.name").alias("name")
    )
    .select(["id", "name", "label", "properties"])
    .sort("id")
)

# Save the table
output_file = Path("data/gold/evals/duplicated_nodes.csv")
if duplicated_nodes_table.height > 0:
    duplicated_nodes_table.write_csv(output_file)
    print(f"Found {duplicate_ids.height} unique duplicated IDs resulting in {duplicated_nodes_table.height} total rows.")
    print(f"Table saved to: {output_file}")

# Display the first few rows for inspection
duplicated_nodes_table.head(10)

id,name,label,properties
str,str,str,str


## Neighbor Consistency Check

In this section we compare, for a given set of node IDs, the neighbors reported by the in-memory NetworkX graph `G` against neighbors obtained by directly filtering the `edges.parquet` file. This is useful to debug potential discrepancies between the graph representation and the underlying edge table.

In [ ]:
# Ensure edges dataframe is loaded
edges_path = Path("data/gold/kg/parquet/edges.parquet")
edges_df = pl.read_parquet(edges_path)


def get_out_neighbors_from_graph(G, node_id: str) -> set[str]:
    """Outgoing neighbors of `node_id` from a NetworkX DiGraph.

    Returns nodes reached by edges node_id -> neighbor.
    If the node is not in the graph, returns an empty set.
    """
    if node_id not in G:
        return set()
    return set(G.successors(node_id))


def get_in_neighbors_from_graph(G, node_id: str) -> set[str]:
    """Incoming neighbors of `node_id` from a NetworkX DiGraph.

    Returns nodes with edges neighbor -> node_id.
    If the node is not in the graph, returns an empty set.
    """
    if node_id not in G:
        return set()
    return set(G.predecessors(node_id))


def get_out_neighbors_from_edges_df(edges_df: pl.DataFrame, node_id: str) -> set[str]:
    """Outgoing neighbors of `node_id` from the edges table.

    Returns nodes reached by edges node_id -> neighbor.
    """
    return set(
        edges_df
        .filter(pl.col("from") == node_id)
        .select("to")
        .to_series()
        .to_list()
    )


def get_in_neighbors_from_edges_df(edges_df: pl.DataFrame, node_id: str) -> set[str]:
    """Incoming neighbors of `node_id` from the edges table.

    Returns nodes with edges neighbor -> node_id.
    """
    return set(
        edges_df
        .filter(pl.col("to") == node_id)
        .select("from")
        .to_series()
        .to_list()
    )


def compare_neighbors_for_nodes(
    G,
    edges_df: pl.DataFrame,
    node_ids: list[str],
) -> pl.DataFrame:
    """Compare incoming and outgoing neighbors from `G` vs. `edges_df`.

    For each node, returns:
    - outgoing counts from each source
    - outgoing intersection / mismatches
    - incoming counts from each source
    - incoming intersection / mismatches
    """
    records: list[dict] = []

    for node_id in node_ids:
        graph_out = get_out_neighbors_from_graph(G, node_id)
        graph_in = get_in_neighbors_from_graph(G, node_id)

        df_out = get_out_neighbors_from_edges_df(edges_df, node_id)
        df_in = get_in_neighbors_from_edges_df(edges_df, node_id)

        out_only_in_graph = sorted(graph_out - df_out)
        out_only_in_edges = sorted(df_out - graph_out)
        out_intersection = graph_out & df_out
        out_mismatch = graph_out != df_out

        in_only_in_graph = sorted(graph_in - df_in)
        in_only_in_edges = sorted(df_in - graph_in)
        in_intersection = graph_in & df_in
        in_mismatch = graph_in != df_in

        records.append(
            {
                "node_id": node_id,

                "graph_out_neighbors_count": len(graph_out),
                "edges_out_neighbors_count": len(df_out),
                "out_intersection_count": len(out_intersection),
                "out_mismatch": out_mismatch,
                "out_only_in_graph": " ".join(out_only_in_graph),
                "out_only_in_edges": " ".join(out_only_in_edges),

                "graph_in_neighbors_count": len(graph_in),
                "edges_in_neighbors_count": len(df_in),
                "in_intersection_count": len(in_intersection),
                "in_mismatch": in_mismatch,
                "in_only_in_graph": " ".join(in_only_in_graph),
                "in_only_in_edges": " ".join(in_only_in_edges),
            }
        )

    return pl.DataFrame(records)


# Example usage
# Set seed
random.seed(42)
random_ids = random.sample(list(G.nodes), 1000)
comparison_result = compare_neighbors_for_nodes(G, edges_df, random_ids)

# Save to CSV
comparison_result.write_csv("data/gold/evals/neighbor_comparison.csv")
comparison_result

node_id,graph_out_neighbors_count,edges_out_neighbors_count,out_intersection_count,out_mismatch,out_only_in_graph,out_only_in_edges,graph_in_neighbors_count,edges_in_neighbors_count,in_intersection_count,in_mismatch,in_only_in_graph,in_only_in_edges
str,i64,i64,i64,bool,str,str,i64,i64,i64,bool,str,str
"""MONDO_0017748""",260,260,260,false,"""""","""""",255,3,3,true,"""ENSG00000004848 ENSG0000000547…",""""""
"""EFO_0007289""",13,12,12,true,"""CHEMBL685""","""""",14,2,2,true,"""ENSG00000102245 ENSG0000013455…",""""""
"""ENSG00000227011""",38,0,0,true,"""UBERON_0000006 UBERON_0000178 …","""""",38,38,38,false,"""""",""""""
"""CHEMBL3039743""",1,1,1,false,"""""","""""",0,0,0,false,"""""",""""""
"""MONDO_0958006""",121,121,121,false,"""""","""""",122,1,1,true,"""ENSG00000001630 ENSG0000002955…",""""""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""MONDO_0002739""",2,2,2,false,"""""","""""",2,2,2,false,"""""",""""""
"""CHEMBL5095446""",1,1,1,false,"""""","""""",1,0,0,true,"""MONDO_0009861""",""""""
"""UBERON_0034678""",0,0,0,false,"""""","""""",1,1,1,false,"""""",""""""


In [24]:
len(set(G.successors("EFO_0007289")))

13

In [19]:
edges_df.filter(pl.col("from") == "EFO_0007289")

from,to,label,relation,undirected,properties
str,str,str,str,bool,str
"""EFO_0007289""","""ENSG00000102245""","""DIS-PRO""","""ASSOCIATED_WITH""",true,"""{""evidence_score"":0.0086851265…"
"""EFO_0007289""","""ENSG00000134551""","""DIS-PRO""","""ASSOCIATED_WITH""",true,"""{""evidence_score"":0.0014783194…"
"""EFO_0007289""","""ENSG00000137462""","""DIS-PRO""","""ASSOCIATED_WITH""",true,"""{""evidence_score"":0.0014783194…"
"""EFO_0007289""","""ENSG00000137745""","""DIS-PRO""","""ASSOCIATED_WITH""",true,"""{""evidence_score"":0.0022174791…"
"""EFO_0007289""","""ENSG00000138398""","""DIS-PRO""","""ASSOCIATED_WITH""",true,"""{""evidence_score"":0.0044349582…"
…,…,…,…,…,…
"""EFO_0007289""","""ENSG00000197746""","""DIS-PRO""","""ASSOCIATED_WITH""",true,"""{""evidence_score"":0.0014783194…"
"""EFO_0007289""","""ENSG00000198804""","""DIS-PRO""","""ASSOCIATED_WITH""",true,"""{""evidence_score"":0.0014783194…"
"""EFO_0007289""","""ENSG00000211891""","""DIS-PRO""","""ASSOCIATED_WITH""",true,"""{""evidence_score"":0.0184481944…"


In [20]:
edges_df.filter(
    (pl.col("from") == "EFO_0007289") &
    (pl.col("to") == "CHEMBL685")
)

from,to,label,relation,undirected,properties
str,str,str,str,bool,str


In [21]:
edges_df.filter(
    (pl.col("to") == "EFO_0007289") &
    (pl.col("from") == "CHEMBL685")
)

from,to,label,relation,undirected,properties
str,str,str,str,bool,str
"""CHEMBL685""","""EFO_0007289""","""DRG-DIS""","""OFF_LABEL_USE""",true,"""{""sources"":{""direct"":[""DRUG_CE…"


In [22]:
edges_df.filter(
    (pl.col("to") == "EFO_0007289")
)

from,to,label,relation,undirected,properties
str,str,str,str,bool,str
"""EFO_1001342""","""EFO_0007289""","""DIS-DIS""","""PARENT""",false,"""{""sources"":{""direct"":[""OPEN_TA…"
"""CHEMBL685""","""EFO_0007289""","""DRG-DIS""","""OFF_LABEL_USE""",true,"""{""sources"":{""direct"":[""DRUG_CE…"
